# RescueBeacon backend development notebook

The canonical service is `backend/main.py`. This notebook imports it rather than maintaining another implementation. Actual original images go to the supplied GB10 Qwen server; later drone findings remain simulated. For the full UI use `npm run dev` from the repo root.


In [ ]:
%pip install fastapi uvicorn openai httpx


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "images/manifest.json").is_file() and (p / "backend/main.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from backend.main import app, MODEL, MODEL_URL
print("Model:", MODEL, "at", MODEL_URL)


In [ ]:
import asyncio
import uvicorn
if "server_task" not in globals() or server_task.done():
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000))
    server_task = asyncio.create_task(server.serve())
print("API: http://127.0.0.1:8000/docs")


## Real image assessment

This makes a real model request. The second API call consumes the paired assistance judgment, without sending the image again. Use any observation ID from the manifest.


In [ ]:
import httpx
async with httpx.AsyncClient(timeout=100, trust_env=False) as api:
    for stage in ["flood", "assistance"]:
        response = await api.post("http://127.0.0.1:8000/api/assessments", json={"observationId": "drone-clear", "stage": stage})
        print(response.status_code, response.json())


In [ ]:
if "server_task" in globals() and not server_task.done():
    server.should_exit = True
    await asyncio.wait_for(server_task, timeout=100)
